## Modelo predictivo de coste de seguro

## Cargamos el CSV y las librerias

In [26]:
import pandas as pd
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_validate
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error, r2_score

In [27]:
df= pd.read_csv("Data/df_con_probabilidad.csv")

In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101886 entries, 0 to 101885
Data columns (total 24 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   ID                                     101886 non-null  str    
 1   Edad                                   101886 non-null  int64  
 2   Ingresos                               101886 non-null  float64
 3   Monto_Inicial                          101886 non-null  int64  
 4   Scoring_Crediticio                     101886 non-null  int64  
 5   Meses_Empleo                           101886 non-null  float64
 6   Num_Creditos                           101886 non-null  int64  
 7   Ratio_Interes                          101886 non-null  float64
 8   Duracion                               101886 non-null  float64
 9   Ratio_Deuda_Ingresos                   101886 non-null  float64
 10  Estudios                               101886 non-null  int64  
 11

In [29]:
df= df.drop(columns="Impago")

Eliminamos la columna impago porque es una clasificacion poco realista, Esto es debido a que la clasificacion depende de un umbral que nosotros mismos hemos elegido a proposito para que trate de detectar el maximo numero de impagos posibles. Es por eso que vamos a trabajar unicamente con la probabilidad de impago que nuestro modelo nos ha devuelto

Aun asi viendo las variables me he dado cuenta que a la hora de hacer un get dummies de las variables categoricas se nos olvido eliminar una columna por lo que vamos a eliminar una de cada una. Esto es necesario porque de no hacerlo la suma de las categorias siempre sera 1 porque imaginate si tengo tres categorias (en formato dummy) y se dos de ellas no necesito la tercera 

Para ello he decidido eliminar la mas frecuente en ambas para que sea una referencia intuitiva

In [30]:
cols_estado_civil = ["Estado_Civil_Casado", "Estado_Civil_Divorciado", "Estado_Civil_Soltero"]
cols_jornada = ["Tipo_Jornada_Laboral_Autónomo", "Tipo_Jornada_Laboral_Desempleado",
                "Tipo_Jornada_Laboral_Jornada completa", "Tipo_Jornada_Laboral_Tiempo parcial"]

In [31]:
estado_civil_media = df[cols_estado_civil].mean()
print(estado_civil_media)

Estado_Civil_Casado        0.511778
Estado_Civil_Divorciado    0.186424
Estado_Civil_Soltero       0.301798
dtype: float64


In [32]:
df= df.drop(columns="Estado_Civil_Casado")

In [33]:
jornada_media = df[cols_jornada].mean()
print(jornada_media)

Tipo_Jornada_Laboral_Autónomo            0.148519
Tipo_Jornada_Laboral_Desempleado         0.099513
Tipo_Jornada_Laboral_Jornada completa    0.550056
Tipo_Jornada_Laboral_Tiempo parcial      0.201912
dtype: float64


In [34]:
df= df.drop(columns="Tipo_Jornada_Laboral_Jornada completa")

## Separamos en TRAIN TEST y PUBLICO

In [35]:
df_modelo = df.copy()

df_modelo = df_modelo.drop(columns="ID")


df_publico = df_modelo.sample(frac=0.10, random_state=42)
df_modelo = df_modelo.drop(df_publico.index)


target = "Prima"
X = df_modelo.drop(columns=target)
y = df_modelo[target]

X_publico = df_publico.drop(columns=target)
y_publico = df_publico[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Regresion simple

In [36]:
columnas_a_escalar = ["Edad", "Ingresos", "Monto_Inicial", "Scoring_Crediticio",
                      "Num_Creditos", "Ratio_Interes", "Duracion", "Ratio_Deuda_Ingresos"]

scaler = StandardScaler()

X_train_scal = X_train.copy()
X_test_scal = X_test.copy()
X_publico_scal = X_publico.copy()

X_train_scal[columnas_a_escalar] = scaler.fit_transform(X_train[columnas_a_escalar])
X_test_scal[columnas_a_escalar] = scaler.transform(X_test[columnas_a_escalar])
X_publico_scal[columnas_a_escalar] = scaler.transform(X_publico[columnas_a_escalar])

In [37]:
X_train_scaled = sm.add_constant(X_train_scal, prepend=True)
modelo_lin_reg = sm.OLS(endog=y_train, exog=X_train_scaled)
modelo_lin_reg = modelo_lin_reg.fit()
print(modelo_lin_reg.summary())

                            OLS Regression Results                            
Dep. Variable:                  Prima   R-squared:                       0.677
Model:                            OLS   Adj. R-squared:                  0.677
Method:                 Least Squares   F-statistic:                     8102.
Date:                Sun, 01 Nov 2026   Prob (F-statistic):               0.00
Time:                        12:39:45   Log-Likelihood:            -4.7248e+05
No. Observations:               73357   AIC:                         9.450e+05
Df Residuals:                   73337   BIC:                         9.452e+05
Df Model:                          19                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
co

Vamos a quitar las variables con un p valor mas alto que 0,05 ya que si este es mas alto que 0,05 no podriamos rechazar la hipotesis nula y por lo tanto esas variables no serian estadisticamente significativas

In [38]:
variables_quitar = [
    "Meses_Empleo",
    "Num_Creditos",
    "Posesion_Hipoteca",
    "Personas_Cargo",
    "Fiador",
    "Estado_Civil_Divorciado",
    "Estado_Civil_Soltero",
    "Tipo_Jornada_Laboral_Autónomo",
    "Tipo_Jornada_Laboral_Desempleado",
    "Tipo_Jornada_Laboral_Tiempo parcial"
    
]

In [39]:
X_train_hipotesisnula =X_train_scal.drop(columns=variables_quitar)
X_test_hipotesisnula = X_test_scal.drop(columns=variables_quitar)

In [40]:
X_train_scaled = sm.add_constant(X_train_hipotesisnula, prepend=True)
modelo_lin_reg = sm.OLS(endog=y_train, exog=X_train_scaled)
modelo_lin_reg = modelo_lin_reg.fit()
print(modelo_lin_reg.summary())

                            OLS Regression Results                            
Dep. Variable:                  Prima   R-squared:                       0.677
Model:                            OLS   Adj. R-squared:                  0.677
Method:                 Least Squares   F-statistic:                 1.711e+04
Date:                Sun, 01 Nov 2026   Prob (F-statistic):               0.00
Time:                        12:39:45   Log-Likelihood:            -4.7248e+05
No. Observations:               73357   AIC:                         9.450e+05
Df Residuals:                   73347   BIC:                         9.451e+05
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                  424.6818 

## Regresion Polinomica

In [44]:
def comparar_grados(X, y, grados=(3,4, 5)):

    # Definimos cómo se divide el train en la validación cruzada:
    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    # Lista donde iremos guardando los resultados de cada grado
    resultados = []

    # Repetimos el mismo proceso para cada grado del polinomio
    for g in grados:
        # Generamos las variables polinómicas
        # include_bias=False evita añadir una columna de unos, porque LinearRegression ya calcula el intercepto
        poly = PolynomialFeatures(degree=g, include_bias=False)
        X_poly = poly.fit_transform(X)

        # Validación cruzada de la regresión lineal sobre las variables polinómicas:
        # return_train_score=True guarda el error sobre los folds de entrenamiento
        # scoring calcula varias métricas a la vez (las de error vienen en negativo porque scikit-learn siempre maximiza)
        scores = cross_validate(LinearRegression(), X_poly, y, cv=cv, return_train_score=True,
                                scoring={"rmse": "neg_root_mean_squared_error",
                                         "mae": "neg_mean_absolute_error",
                                         "r2": "r2"})

        # Guardamos un resumen de este grado: la media de cada métrica en los 5 folds
        # El signo negativo delante de los errores los vuelve a poner en positivo
        resultados.append({
            "grado": g,
            "n_variables": X_poly.shape[1],               # número de columnas tras la transformación polinómica
            "RMSE_train": -scores["train_rmse"].mean(),   # error sobre los datos de train
            "RMSE_cv": -scores["test_rmse"].mean(),       # error sobre los folds no vistos(el que se usa de test)
            "MAE_cv": -scores["test_mae"].mean(),         # error absoluto medio en test (en gramos)
            "R2_cv": scores["test_r2"].mean()             # R cuadrado en test
        })

    # Convertimos la lista de resultados en una tabla con una fila por grado
    return pd.DataFrame(resultados)

# Aplicamos la función solo sobre el conjunto de entrenamiento (el test se reserva para la evaluación final)
resultados = comparar_grados(X_train_hipotesisnula, y_train)
resultados

,grado,n_variables,RMSE_train,RMSE_cv,MAE_cv,R2_cv
0,3,219,74.580196,74.996857,55.822367,0.921121
1,4,714,63.302325,64.363331,47.005303,0.941903
2,5,2001,56.861266,59.928520,42.573170,0.949635


In [45]:
mejor_grado = int(resultados.loc[resultados["RMSE_cv"].idxmin(), "grado"])

# Generamos las variables polinómicas con el grado elegido
poly = PolynomialFeatures(degree=mejor_grado, include_bias=False)
X_train_poly = poly.fit_transform(X_train_hipotesisnula)  
X_test_poly = poly.transform(X_test_hipotesisnula)        

# Entrenamos el modelo final con todo el conjunto de entrenamiento
modelo_final = LinearRegression()
modelo_final.fit(X_train_poly, y_train)

# Predecimos sobre el test: datos que el modelo no ha visto en ningún momento
pred_test = modelo_final.predict(X_test_poly)

# Evaluamos el modelo una única vez en test: esta es la estimación final de su rendimiento con datos nuevos
print(f"Mejor grado: {mejor_grado}")
print(f"Test:  RMSE = {root_mean_squared_error(y_test, pred_test):.2f} | "   
      f"MAE = {mean_absolute_error(y_test, pred_test):.2f} | "               
      f"R² = {r2_score(y_test, pred_test):.3f}")    

Mejor grado: 5
Test:  RMSE = 58.46 | MAE = 41.86 | R² = 0.952
